In [1]:
# --- PARÂMETROS DE ENTRADA ---
ticker = "WDO$"  # Ativo para o replay (deve estar no main.yaml)
replay_datetime_str = "2025-10-01 09:00" # Data ('YYYY-MM-DD') ou Data e Hora ('YYYY-MM-DD HH:MM')
# ---------------------------

print(f"Configurado para análise do ativo '{ticker}' em '{replay_datetime_str}'.")

Configurado para análise do ativo 'WDO$' em '2025-10-01 09:00'.


In [2]:
import yaml
import logging
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
import pytz
import MetaTrader5 as mt5
import mplfinance as mpf
import sys
import base64
from io import BytesIO
import matplotlib.pyplot as plt
import importlib

# Adiciona a pasta 'src' ao path
project_root = Path.cwd().parent.parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.strategies.lstm import KerasLSTMWrapper

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# Carrega as configurações
with open(project_root / "configs/main.yaml", "r") as file:
    config = yaml.safe_load(file)

asset_config = next((asset for asset in config['assets'] if asset['ticker'] == ticker), None)
if not asset_config:
    raise ValueError(f"Ticker '{ticker}' não encontrado em configs/main.yaml.")

In [3]:
# 1. Processar data e hora
datetime_str = replay_datetime_str

try:
    end_dt_obj = datetime.strptime(datetime_str, '%Y-%m-%d %H:%M')
    replay_day_start = end_dt_obj.replace(hour=0, minute=0, second=0)
except ValueError:
    replay_day_start = datetime.strptime(datetime_str, '%Y-%m-%d')
    end_dt_obj = replay_day_start.replace(hour=23, minute=59, second=59)

fetch_start_obj = replay_day_start - timedelta(days=6)
timezone = pytz.timezone("Etc/UTC")
start_time_utc = timezone.localize(fetch_start_obj)
end_time_utc = timezone.localize(end_dt_obj)

# 2. Buscar dados de candles M5
logging.info(f"Conectando ao MT5 para buscar candles de M5 de {start_time_utc.date()} até {end_time_utc.date()}...")
if not mt5.initialize():
    logging.error(f"Falha na inicialização do MT5: {mt5.last_error()}")
    sys.exit(1)

rates = mt5.copy_rates_range(ticker, mt5.TIMEFRAME_M5, start_time_utc, end_time_utc)
mt5.shutdown()

if rates is None or len(rates) == 0:
    logging.error("Nenhum dado de candle encontrado para o período.")
    sys.exit(1)

full_candles_df = pd.DataFrame(rates)
full_candles_df['time'] = pd.to_datetime(full_candles_df['time'], unit='s')
full_candles_df.set_index('time', inplace=True)
full_candles_df.rename(columns={'tick_volume': 'volume'}, inplace=True)
logging.info(f"{len(full_candles_df)} candles de M5 carregados no total.")

# 3. Calcular indicadores
full_candles_df['sma9'] = full_candles_df['close'].rolling(window=9).mean()
full_candles_df['ema21'] = full_candles_df['close'].ewm(span=21, adjust=False).mean()
full_candles_df['ema50'] = full_candles_df['close'].ewm(span=50, adjust=False).mean()
full_candles_df['ema200'] = full_candles_df['close'].ewm(span=200, adjust=False).mean()

# 4. Gerar Relatório de Sinais
replay_day_candles_df = full_candles_df[full_candles_df.index.date == replay_day_start.date()].copy()

if replay_day_candles_df is None:
    logging.error("Erro ao coletar dados históricos. Encerrando.")
    sys.exit(1)

logging.info("Dados Coletados...")

2025-10-12 16:50:47,836 - INFO - Conectando ao MT5 para buscar candles de M5 de 2025-09-25 até 2025-10-01...
2025-10-12 16:50:47,853 - INFO - 457 candles de M5 carregados no total.
2025-10-12 16:50:47,858 - INFO - Dados Coletados...


In [8]:
current_candle_time = replay_day_candles_df.index[-1]
data_for_prediction = full_candles_df[full_candles_df.index <= current_candle_time]

print(f"Current Candle Time: {current_candle_time}  Data for Prediction Length: {len(data_for_prediction)}")

Current Candle Time: 2025-10-01 09:00:00  Data for Prediction Length: 457


In [ ]:
#busca dados 

,open,high,low,close,volume,spread,real_volume,sma9,ema21,ema50,ema200
time,,,,,,,,,,,
2025-09-30 10:15:00,5352.5,5357.0,5351.0,5355.5,8311,1,29524,5357.555556,5358.037771,5360.126464,5369.455950
2025-09-30 10:20:00,5356.0,5358.0,5354.5,5357.0,5992,1,22175,5358.500000,5357.943429,5360.003857,5369.332010
2025-09-30 10:25:00,5356.5,5360.0,5356.0,5359.0,6482,1,26880,5358.611111,5358.039481,5359.964490,5369.229204
2025-09-30 10:30:00,5359.0,5359.0,5352.0,5354.0,8979,1,35821,5357.500000,5357.672255,5359.730589,5369.077670
2025-09-30 10:35:00,5353.5,5356.5,5352.5,5355.5,3889,1,20510,5356.611111,5357.474777,5359.564683,5368.942568
...,...,...,...,...,...,...,...,...,...,...,...
2025-09-30 18:10:00,5363.0,5364.0,5361.5,5361.5,758,1,2656,5362.333333,5362.933678,5363.355596,5365.821169
2025-09-30 18:15:00,5362.0,5363.5,5360.0,5361.5,984,1,4076,5362.333333,5362.803343,5363.282827,5365.778172
2025-09-30 18:20:00,5361.0,5363.0,5360.0,5360.5,1042,1,3996,5362.166667,5362.593949,5363.173697,5365.725653


In [ ]:
models_dir = project_root / config["global_settings"]["model_directory"]
model_path = models_dir / f"{ticker}_prod_model.keras"
scaler_path = models_dir / f"{ticker}_prod_scaler.joblib"

if not model_path.exists():
    logging.error(f"Modelo para {ticker} não encontrado.")
    sys.exit(1)
    
model = KerasLSTMWrapper.load_model(str(model_path), str(scaler_path))
strategy_module = importlib.import_module(f"src.strategies.{asset_config['strategy_module']}")
StrategyClass = getattr(strategy_module, asset_config['strategy_name'])
strategy = StrategyClass()    

featured_data = strategy.define_features(data_for_prediction)
X_live = featured_data[strategy.get_feature_names()].dropna()

if X_live.empty: 
    print("Dados insuficientes para gerar features.")
    sys.exit(1)

signal = model.predict(X_live)[-1]
entry_price = replay_day_candles_df['close'].iloc[-1]
stop_loss_pct = asset_config['trading_rules']['stop_loss_pct']
stop_price = entry_price * (1 - stop_loss_pct) if signal == 1 else entry_price * (1 + stop_loss_pct)

sugestao = []

sugestao.append({
    "Tipo de Operação": "Compra" if signal == 1 else "Venda",
    "Candle de Entrada": current_candle_time.strftime('%Y-%m-%d %H:%M'),
    "Preço Sugerido": f"{entry_price:.2f}",
    "Preço de Stop": f"{stop_price:.2f}",
    "signal": signal, "timestamp": current_candle_time
})

print(f"Posição sugerida: {position}")

2025-10-12 16:50:47,917 - INFO - Carregando modelo de c:\projects\wtnps-trade\models\WDO$_prod_model.keras e scaler de c:\projects\wtnps-trade\models\WDO$_prod_scaler.joblib
c:\Users\User\AppData\Local\pypoetry\Cache\virtualenvs\wtnps-trade-VyqtAXyS-py3.12\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step
Posição sugerida: [{'Tipo de Operação': 'Compra', 'Candle de Entrada': '2025-10-01 09:00', 'Preço Sugerido': '5343.50', 'Preço de Stop': '5236.63', 'signal': np.int64(1), 'timestamp': Timestamp('2025-10-01 09:00:00')}]
